# Agentic Log Anomaly Explanation Pipeline

## Complete Walkthrough

This notebook provides a clean, end-to-end walkthrough of the **Screener-Reasoner** pipeline for generating traceable explanations of log anomalies.


## 1. Setup & Imports

In [1]:
# Standard library
import sys
import json
import time
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.data_loader import BGLDataLoader, Session
from src.screener import Screener, ScreenerOutput
from src.evidence_store import EvidenceStore, EvidenceDoc
from src.retriever import BM25Retriever, Retriever
from src.prompt_builder import PromptBuilder, TraceExplanation, Claim, Signature, format_evidence_block
from src.llm_client import LLMClient
from src.config_loader import load_config, get_llm_kwargs
from src.verifier import Verifier
from src.normalizer import get_normalizer

print("✓ All imports successful")

✓ All imports successful


## 2. Data Loading

Load the BGL (Blue Gene/L) supercomputer log dataset.

- **Train split**: Used to build evidence store (never seen during inference)
- **Test split**: Sessions to detect and explain anomalies

In [2]:
# Load BGL dataset
loader = BGLDataLoader(log_file="../logs/BGL.log")
loader.load()
loader.print_stats()

# Get splits
train_sessions = loader.get_train()
test_sessions = loader.get_test()

print(f"\nTrain sessions: {len(train_sessions):,}")
print(f"Test sessions: {len(test_sessions):,}")

# Count anomalies in each split
train_anomaly = sum(1 for s in train_sessions if s.label == 1)
test_anomaly = sum(1 for s in test_sessions if s.label == 1)
print(f"\nTrain anomalies: {train_anomaly:,} ({train_anomaly/len(train_sessions):.1%})")
print(f"Test anomalies: {test_anomaly:,} ({test_anomaly/len(test_sessions):.1%})")

Loading BGL logs from: ../logs/BGL.log


Reading BGL logs: 4747963it [00:01, 3850983.10it/s]


Loaded 4747963 log lines


Creating sessions: 100%|██████████| 474796/474796 [00:02<00:00, 182243.84it/s]



BGL Dataset Statistics

TRAIN:
  Total sessions: 332,356
  Normal: 305,041 | Anomaly: 27,315
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

VAL:
  Total sessions: 71,219
  Normal: 65,366 | Anomaly: 5,853
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

TEST:
  Total sessions: 71,221
  Normal: 65,367 | Anomaly: 5,854
  Anomaly ratio: 8.22%
  Avg lines/session: 10.0

Train sessions: 332,356
Test sessions: 71,221

Train anomalies: 27,315 (8.2%)
Test anomalies: 5,854 (8.2%)


### Example Session

Each session is a group of log lines with a label (0=normal, 1=anomaly).

In [3]:
# Show an example anomalous session
example_anomaly = next(s for s in test_sessions if s.label == 1)

print(f"Session ID: {example_anomaly.session_id}")
print(f"Label: {'ANOMALY' if example_anomaly.label == 1 else 'NORMAL'}")
print(f"Lines: {len(example_anomaly.lines)}")
print("\nFirst 5 lines:")
for i, line in enumerate(example_anomaly.lines[:5], 1):
    print(f"  {i}. {line[:100]}..." if len(line) > 100 else f"  {i}. {line}")

Session ID: BGL_00442780
Label: ANOMALY
Lines: 10

First 5 lines:
  1. 1118765577 2005.06.14 R04-M0-N4-C:J14-U11 2005-06-14-09.12.57.265757 R04-M0-N4-C:J14-U11 RAS KERNEL ...
  2. 1118765577 2005.06.14 R04-M0-N4-C:J10-U11 2005-06-14-09.12.57.295068 R04-M0-N4-C:J10-U11 RAS KERNEL ...
  3. 1118765577 2005.06.14 R04-M0-N4-C:J06-U11 2005-06-14-09.12.57.376176 R04-M0-N4-C:J06-U11 RAS KERNEL ...
  4. 1118765577 2005.06.14 R04-M0-N4-C:J12-U11 2005-06-14-09.12.57.406016 R04-M0-N4-C:J12-U11 RAS KERNEL ...
  5. 1118765577 2005.06.14 R04-M0-N4-C:J14-U01 2005-06-14-09.12.57.433568 R04-M0-N4-C:J14-U01 RAS KERNEL ...


## 3. Screener (Anomaly Detection)

The Screener is a neural network (AllLinLog) that classifies log sessions as normal or anomalous.

**Output:**
- `pred`: Binary prediction (0=normal, 1=anomaly)
- `prob`: Probability distribution [P(normal), P(anomaly)]
- `margin`: Confidence measure (|P(anomaly) - P(normal)|)

In [4]:
# Load pre-trained screener model
screener = Screener.from_pretrained(
    dataset="BGL",
    model_path="../best_model/best_model_20250724_072857.pth"
)

total_params = sum(p.numel() for p in screener.model.parameters())
print(f"Model parameters: {total_params:,}")

Loading Screener for BGL on cuda
Loading cl100k_base (GPT-4) tokenizer...
Loading model weights from: ../best_model/best_model_20250724_072857.pth
Model loaded! Parameters: 13,445,922
Model parameters: 13,445,922


In [5]:
# Screen a sample of test sessions
sample_size = 50
sample_sessions = test_sessions[:sample_size]

print(f"Screening {sample_size} sessions...")
start = time.time()
screener_outputs = screener.screen_sessions(sample_sessions)
elapsed = time.time() - start

print(f"Done in {elapsed:.2f}s ({elapsed/sample_size*1000:.1f}ms per session)")

# Count predictions
predicted_anomalies = [o for o in screener_outputs if o.is_anomaly]
print(f"\nPredicted anomalies: {len(predicted_anomalies)} / {sample_size}")

Screening 50 sessions...


Screening sessions: 100%|██████████| 7/7 [00:00<00:00, 36.54it/s]

Done in 0.19s (3.9ms per session)

Predicted anomalies: 2 / 50


In [6]:
# Show a screener output example
if predicted_anomalies:
    output = predicted_anomalies[0]
    print("Example Screener Output:")
    print(f"  Session ID: {output.session_id}")
    print(f"  Prediction: {'ANOMALY' if output.is_anomaly else 'NORMAL'}")
    print(f"  Anomaly Probability: {output.anomaly_prob:.2%}")
    print(f"  Confidence Margin: {output.margin:.4f}")

Example Screener Output:
  Session ID: BGL_00442780
  Prediction: ANOMALY
  Anomaly Probability: 100.00%
  Confidence Margin: 1.0000


## 4. Evidence Store

The Evidence Store indexes all training sessions for RAG retrieval.

**Key features:**
- Stores normalized text for each session
- Tracks label (normal/anomaly) for mixed retrieval
- Supports multiple evidence types (session, signature)
- Signature cards loaded from `patterns/bgl_patterns.json` (34 data-driven patterns)

In [7]:
# Build evidence store from training sessions
evidence_store = EvidenceStore(dataset="BGL")
evidence_store.build_from_sessions(train_sessions, show_progress=True)

# Show statistics
stats = evidence_store.stats()
print("\nEvidence Store Stats:")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

Building evidence store: 100%|██████████| 332356/332356 [01:31<00:00, 3648.19it/s]


Evidence store built with 332356 documents

Evidence Store Stats:
  total_documents: 332,356
  normal_documents: 305,041
  anomaly_documents: 27,315
  by_evidence_type: {'session': 332356, 'signature': 0, 'profile': 0}
  avg_text_length: 1541.498495589067
  min_text_length: 969
  max_text_length: 6,719


### Add Signature Cards

Signature cards are loaded from `patterns/bgl_patterns.json` — 34 data-driven patterns discovered by `05_signature_audit.ipynb`.

In [8]:
# Load pre-discovered BGL patterns (34 patterns from data-driven signature audit)
# Generated by 05_signature_audit.ipynb, saved in patterns/ directory
patterns_file = Path("..") / "patterns" / "bgl_patterns.json"
with open(patterns_file, 'r') as f:
    bgl_patterns = json.load(f)

# Add each pattern as a signature card to evidence store
for pattern_id, pattern_info in bgl_patterns.items():
    # BGL uses 'fingerprint', HDFS uses 'merge_key'
    pattern_key = pattern_info.get('merge_key', pattern_info.get('fingerprint', 'N/A'))
    sig_text = f"""ERROR SIGNATURE: {pattern_info['name']}
Description: {pattern_info['description']}

Key Indicators: {', '.join(pattern_info['keywords'])}
Frequency: {pattern_info['frequency']} occurrences in training data

Pattern Characteristics:
  - Fingerprint: {pattern_key}
"""
    
    doc = EvidenceDoc(
        evidence_id=f"E_SIG_{pattern_id}",
        session_id=pattern_id,
        text=sig_text,
        evidence_type="signature",
        metadata={
            "label": 1,
            "dataset": "BGL",
            "signature_name": pattern_info['name'],
            "frequency": pattern_info['frequency'],
            "keywords": pattern_info['keywords'],
        }
    )
    evidence_store.documents.append(doc)
    evidence_store._id_to_doc[doc.evidence_id] = doc

print(f"Loaded {len(bgl_patterns)} BGL signature cards from {patterns_file.resolve()}")
print(f"Evidence store now has {len(evidence_store.documents):,} documents")

# Top 5 by frequency
sorted_patterns = sorted(bgl_patterns.items(), key=lambda x: x[1]['frequency'], reverse=True)[:5]
for pid, p in sorted_patterns:
    print(f"  - {p['name'][:60]}: {p['frequency']:,} sessions")

Loaded 34 BGL signature cards from /home/dave/agentic-log-explanations/patterns/bgl_patterns.json
Evidence store now has 332,390 documents
  - KERNEL__DATA_TLB_ERROR: 11,052 sessions
  - KERNEL__DATA_STORAGE_INTERRUPT: 5,196 sessions
  - APP__CIOD_SOCKET_ERROR: 4,450 sessions
  - KERNEL__FATAL_ERROR: 3,941 sessions
  - KERNEL__HARDWARE_ERROR: 668 sessions


## 5. Retriever (RAG)

The Retriever finds similar historical sessions using BM25.

### Mixed Retrieval (Phase 2)

To support **contrast claims** ("unlike normal sessions..."), we retrieve:
- **4 anomaly** sessions (for pattern matching)
- **1 normal** session (for contrast)

In [9]:
# Build BM25 retriever
retriever = BM25Retriever(evidence_store)
retriever.build_index()

Building BM25 index...
BM25 index built with 332390 documents


In [10]:
# Test mixed retrieval on an anomalous session
test_session = sample_sessions[screener_outputs.index(predicted_anomalies[0])]

# Standard retrieval (top-5 any label)
print("=== Standard Retrieval (top-5 any) ===")
standard_hits = retriever.retrieve_for_session(test_session, top_k=5)
for h in standard_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

# Mixed retrieval (4 anomaly + 1 normal)
print("\n=== Mixed Retrieval (4 anomaly + 1 normal) ===")
mixed_hits = retriever.retrieve_for_session_mixed(test_session, top_k_anomaly=4, top_k_normal=1)
for h in mixed_hits:
    label = "anomaly" if h.metadata.get("label") == 1 else "normal"
    print(f"  {h.evidence_id[:25]:25s} label={label:7s} score={h.score:.2f}")

=== Standard Retrieval (top-5 any) ===
  E_BGL_00383860            label=anomaly score=630.89
  E_BGL_00392820            label=anomaly score=630.89
  E_BGL_00499060            label=anomaly score=630.89
  E_BGL_00486760            label=anomaly score=630.89
  E_BGL_00429330            label=anomaly score=630.89

=== Mixed Retrieval (4 anomaly + 1 normal) ===
  E_BGL_00383860            label=anomaly score=630.89
  E_BGL_00392820            label=anomaly score=630.89
  E_BGL_00499060            label=anomaly score=630.89
  E_BGL_00486760            label=anomaly score=630.89
  E_BGL_00619630            label=normal  score=528.79


## 6. Prompt Builder

The Prompt Builder formats the session and evidence into a structured prompt.

### Evidence ID Convention
- `[E0]` = Query session (the session being analyzed)
- `[E1]`, `[E2]`, ... = Retrieved historical evidence

### Claim Types
| Type | Description | Example |
|------|-------------|----------|
| `observation` | Direct observation from E0 | "E0 contains KERNEL FATAL errors" |
| `pattern_match` | Matches known anomaly | "E0 pattern matches E1, E2" |
| `contrast` | Differs from normal | "Unlike E5 (normal), E0 shows FATAL" |

In [11]:
# Initialize prompt builder
builder = PromptBuilder(
    # Dynamic log display (no hard line limit),
    max_chars_per_evidence=50000,
    max_evidence_items=5
)

# Create a mock screener output for the test session
scr_output = predicted_anomalies[0]

# Build prompt
system_prompt, user_prompt = builder.build_prompt(
    session=test_session,
    screener_output=scr_output,
    evidence_hits=mixed_hits
)

print("=== SYSTEM PROMPT===")
print(system_prompt)
print("...")

print("\n=== USER PROMPT ===")
print(user_prompt)
print("...")

=== SYSTEM PROMPT===
You are an expert log analyst producing forensic, evidence-grounded explanations.
Your task is to analyze a log session flagged as anomalous by an ML screener
(F1-score = 0.996) and provide an evidence-grounded explanation.
The screener is almost always correct — trust its verdict by default.
You may override it ONLY with very high confidence that the session is normal.

DATASET: BlueGene/L supercomputer RAS (Reliability, Availability, Serviceability) logs
COMPONENTS IN LOGS: KERNEL, APP, MMCS, LINKCARD

EVIDENCE FORMAT:
- Each evidence block has LINE NUMBERS: E0-L1, E0-L2, E1-L1, E1-L2, etc.
- [E0] = The query session being analyzed
- [E1]-[E4] = Retrieved anomaly exemplars from historical corpus (for pattern matching)
- [E5] = A NORMAL (non-anomalous) session, provided specifically for contrast claims.
  Use E5 to show how E0 differs from normal behavior.

CLAIM TYPES (you MUST produce at least one of each type when evidence allows):
- "observation": Direct obser

## 7. LLM Client

Call the LLM (GPT-5.1 via OpenAI) to generate an explanation.

In [12]:
# LLM settings are read from configs/config.yaml
# To switch model/provider, edit configs/config.yaml (llm.provider, llm.model, ...)
llm_client = LLMClient(**get_llm_kwargs())

if llm_client.is_available():
    print(f"\u2713 LLM ({llm_client.model}) is available")
else:
    print(f"\u2717 LLM not available. Check API key / provider config.")

✓ LLM (gpt-5.1) is available


In [13]:
# Generate explanation
print("Generating explanation...")
start = time.time()

response = llm_client.generate(
    prompt=user_prompt,
    system_prompt=system_prompt,
    json_mode=True
)

elapsed = time.time() - start
print(f"Done in {elapsed:.2f}s")
print(f"Tokens: {response.total_tokens}")

Generating explanation...
Done in 13.78s
Tokens: 5783


In [14]:
# Parse and display the explanation
explanation_dict = json.loads(response.content)

print("=" * 60)
print("LLM EXPLANATION")
print("=" * 60)

# Signature
if 'signature' in explanation_dict:
    sig = explanation_dict['signature']
    print(f"\nSignature: {sig.get('name', 'N/A')}")

print(f"\nPrediction: {explanation_dict.get('prediction')}")
print(f"Summary: {explanation_dict.get('summary')}")
print(f"\nClaims ({len(explanation_dict.get('claims', []))}):")

for i, claim in enumerate(explanation_dict.get('claims', []), 1):
    print(f"\n  [{i}] {claim.get('type', 'observation')}")
    print(f"      {claim.get('claim', 'N/A')}")
    print(f"      Evidence: {claim.get('evidence_ids', [])} | Spans: {claim.get('evidence_spans', [])}")

LLM EXPLANATION

Signature: KERNEL__DATA_STORAGE_INTERRUPT

Prediction: anomaly
Summary: KERNEL__DATA_STORAGE_INTERRUPT: 10 repeated 'data storage interrupt' FATAL kernel events across multiple nodes in rack R04-M0-N4 within ~0.3 ms at E0-L1 to E0-L10.

Claims (3):

  [1] observation
      E0 contains 10 KERNEL FATAL 'data storage interrupt' events, all on rack R04-M0-N4 nodes (e.g., R04-M0-N4-C:J14-U11, R04-M0-N4-C:J10-U11, R04-M0-N4-C:J04-U01) within timestamps 2005-06-14-09.12.57.265757 to 2005-06-14-09.12.57.574183 at lines E0-L1 to E0-L10.
      Evidence: ['E0'] | Spans: ['E0-L1 to E0-L10']

  [2] pattern_match
      The repeated KERNEL FATAL 'data storage interrupt' pattern in E0 matches the anomaly signature KERNEL__DATA_STORAGE_INTERRUPT seen in historical anomalous sessions, where every line is also 'RAS KERNEL FATAL data storage interrupt' (e.g., E1-L1 to E1-L10).
      Evidence: ['E0', 'E1'] | Spans: ['E0-L1 to E0-L10', 'E1-L1 to E1-L10']

  [3] contrast
      E0 has plain '

## 8. Verifier

The Verifier checks that the explanation is **faithful** to the evidence.

### Verification Checks

| Check | Description |
|-------|-------------|
| `structure` | Required fields present (prediction, summary, claims) |
| `evidence_ids` | All cited evidence IDs are valid |
| `evidence_coverage` | ≥80% of claims cite evidence |
| `keyword_match` | Claim keywords appear in cited evidence |
| `evidence_spans_validity` | Span references are valid |
| `signature` | Signature follows naming format |
| `span_keyword_match` | Keywords appear in cited lines |

In [15]:
# Initialize verifier (min_keyword_match_ratio=0.0 allows LLM abstractions)
verifier = Verifier(min_keyword_match_ratio=0.0)

# Convert dict to TraceExplanation
sig_dict = explanation_dict.get('signature')
signature = None
if sig_dict:
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    )

trace_exp = TraceExplanation(
    prediction=explanation_dict.get('prediction'),
    summary=explanation_dict.get('summary'),
    signature=signature,
    claims=[Claim(
        type=c.get('type', 'observation'),
        claim=c.get('claim'),
        evidence_ids=c.get('evidence_ids', []),
        evidence_spans=c.get('evidence_spans', [])
    ) for c in explanation_dict.get('claims', [])],
    insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
)

# Build evidence ID mapping and verify
evidence_id_mapping = builder.build_evidence_id_mapping(test_session, mixed_hits)
query_session_text = "\n".join(test_session.lines)  # E0 text

verification = verifier.verify(
    explanation=trace_exp,
    evidence_hits=mixed_hits,
    evidence_id_mapping=evidence_id_mapping,
    query_session_text=query_session_text
)

print("=" * 60)
print("VERIFICATION RESULT")
print("=" * 60)
print(f"\nPassed: {verification.passed}")
print(f"Checks: {verification.passed_checks}/{verification.total_checks} passed")

VERIFICATION RESULT

Passed: True
Checks: 9/9 passed


## 9. Complete Pipeline Function

Putting it all together in a single function.

In [16]:
def explain_session(
    session: Session,
    screener_output: ScreenerOutput,
    retriever: BM25Retriever,
    builder: PromptBuilder,
    llm_client: LLMClient,
    verifier: Verifier = None
) -> dict:
    """
    Generate and verify an explanation for an anomalous session.
    """
    start = time.time()
    
    if verifier is None:
        verifier = Verifier(min_keyword_match_ratio=0.0)
    
    # 1. Retrieve evidence (mixed: 4 anomaly + 1 normal)
    evidence_hits = retriever.retrieve_for_session_mixed(
        session, top_k_anomaly=4, top_k_normal=1
    )
    
    # 2. Build prompt
    system_prompt, user_prompt = builder.build_prompt(
        session=session,
        screener_output=screener_output,
        evidence_hits=evidence_hits
    )
    
    # 3. Call LLM
    response = llm_client.generate(
        prompt=user_prompt,
        system_prompt=system_prompt,
        json_mode=True
    )
    
    # 4. Parse response
    try:
        explanation_dict = json.loads(response.content)
        parse_success = True
    except json.JSONDecodeError:
        explanation_dict = {"prediction": "anomaly", "summary": "Parse error", "claims": [], "signature": None}
        parse_success = False
    
    # 5. Convert to TraceExplanation
    sig_dict = explanation_dict.get('signature')
    signature = Signature(
        name=sig_dict.get('name', 'UNKNOWN'),
        matched_evidence_ids=sig_dict.get('matched_evidence_ids', [])
    ) if sig_dict else None
    
    trace_exp = TraceExplanation(
        prediction=explanation_dict.get('prediction'),
        summary=explanation_dict.get('summary'),
        signature=signature,
        claims=[Claim(
            type=c.get('type', 'observation'),
            claim=c.get('claim'),
            evidence_ids=c.get('evidence_ids', []),
            evidence_spans=c.get('evidence_spans', [])
        ) for c in explanation_dict.get('claims', [])],
        insufficient_evidence=explanation_dict.get('insufficient_evidence', False)
    )
    
    # 6. Verify
    evidence_id_mapping = builder.build_evidence_id_mapping(session, evidence_hits)
    query_session_text = "\n".join(session.lines)
    verification = verifier.verify(
        trace_exp, evidence_hits, evidence_id_mapping,
        query_session_text=query_session_text
    )
    
    elapsed = time.time() - start
    
    return {
        'session_id': session.session_id,
        'explanation': explanation_dict,
        'signature': signature.name if signature else None,
        'verification_passed': verification.passed,
        'verification_details': {
            'total_checks': verification.total_checks,
            'passed_checks': verification.passed_checks,
            'failed_checks': verification.failed_checks,
            'warning_checks': verification.warning_checks,
            'issues': [f"{issue.check_name}: {issue.message}" for issue in verification.issues],
        },
        'parse_success': parse_success,
        'tokens': response.total_tokens,
        'latency_ms': elapsed * 1000
    }

print("Pipeline function defined: explain_session()")

Pipeline function defined: explain_session()


In [17]:
# Test the complete pipeline function
result = explain_session(
    session=test_session,
    screener_output=scr_output,
    retriever=retriever,
    builder=builder,
    llm_client=llm_client,
    verifier=verifier
)

print("=" * 60)
print("PIPELINE RESULT")
print("=" * 60)
print(f"Session: {result['session_id']}")
print(f"Parse success: {result['parse_success']}")
print(f"Verification: {'PASSED ✓' if result['verification_passed'] else 'FAILED ✗'}")
print(f"Tokens: {result['tokens']}")
print(f"Latency: {result['latency_ms']:.0f}ms")

PIPELINE RESULT
Session: BGL_00442780
Parse success: True
Verification: PASSED ✓
Tokens: 5769
Latency: 13207ms


## 10. Batch Processing

Process multiple anomalous sessions and collect metrics.

In [18]:
# Batch process predicted anomalies
from tqdm import tqdm
from collections import Counter

# Get all predicted anomalies from screened sessions
anomaly_pairs = [
    (sample_sessions[i], screener_outputs[i])
    for i, o in enumerate(screener_outputs)
    if o.is_anomaly
]

print(f"Processing {len(anomaly_pairs)} predicted anomalies...")

batch_results = []
for session, scr_output in tqdm(anomaly_pairs, desc="Explaining"):
    result = explain_session(
        session=session,
        screener_output=scr_output,
        retriever=retriever,
        builder=builder,
        llm_client=llm_client
    )
    batch_results.append(result)

# Summary statistics
passed = sum(1 for r in batch_results if r['verification_passed'])
total_tokens = sum(r['tokens'] for r in batch_results)
avg_latency = sum(r['latency_ms'] for r in batch_results) / len(batch_results) if batch_results else 0

print("\n" + "=" * 60)
print("BATCH RESULTS")
print("=" * 60)
print(f"\nSessions processed: {len(batch_results)}")
print(f"Verification passed: {passed} / {len(batch_results)} ({passed/len(batch_results):.1%})")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/session: {total_tokens/len(batch_results):.0f}")
print(f"Avg latency: {avg_latency:.0f}ms")

# Signature distribution
signatures = [r['signature'] for r in batch_results if r['signature']]
sig_counts = Counter(signatures)
print(f"\nSignature distribution:")
for sig, count in sig_counts.most_common(10):
    print(f"  {sig}: {count}")

# Show any failed sessions
failed = [r for r in batch_results if not r['verification_passed']]
if failed:
    print(f"\nFailed sessions ({len(failed)}):")
    for r in failed:
        issues = r['verification_details'].get('issues', [])
        print(f"  {r['session_id'][:35]} | {r.get('signature','?')}")
        for issue in issues:
            print(f"    -> {issue}")

Processing 2 predicted anomalies...


Explaining:   0%|          | 0/2 [00:00<?, ?it/s]

Explaining: 100%|██████████| 2/2 [00:34<00:00, 17.35s/it]


BATCH RESULTS

Sessions processed: 2
Verification passed: 2 / 2 (100.0%)
Total tokens: 11,569
Avg tokens/session: 5784
Avg latency: 17343ms

Signature distribution:
  KERNEL__DATA_STORAGE_INTERRUPT: 1
  KERNEL__DATA_TLB_ERROR_INTERRUPT: 1


In [24]:

# Acceptance criteria check
THRESHOLD_PARSE   = 0.96   # parse_success >= 96%
THRESHOLD_VERIFY  = 0.96   # verification_passed >= 96%

if batch_results:
    n = len(batch_results)
    parse_rate  = sum(1 for r in batch_results if r['parse_success']) / n
    verify_rate = sum(1 for r in batch_results if r['verification_passed']) / n

    # LLM error detection (timeout / 403 embedded in failed issues)
    error_keywords = ['timeout', '403', 'ratelimit', 'rate_limit', 'unauthorized']
    llm_errors = [
        r for r in batch_results
        if any(kw in ' '.join(r['verification_details'].get('issues', [])).lower()
               for kw in error_keywords)
    ]

    ok_parse  = parse_rate  >= THRESHOLD_PARSE
    ok_verify = verify_rate >= THRESHOLD_VERIFY
    ok_llm    = len(llm_errors) == 0
    overall   = ok_parse and ok_verify and ok_llm

    print("=" * 60)
    print("ACCEPTANCE CRITERIA (BGL, 100-session smoke test)")
    print("=" * 60)
    print(f"  parse_success  : {parse_rate:.1%}  (threshold >= {THRESHOLD_PARSE:.0%})  {'[PASS]' if ok_parse  else '[FAIL]'}")
    print(f"  verify_passed  : {verify_rate:.1%}  (threshold >= {THRESHOLD_VERIFY:.0%})  {'[PASS]' if ok_verify else '[FAIL]'}")
    print(f"  LLM errors     : {len(llm_errors)}        (threshold = 0)      {'[PASS]' if ok_llm    else '[FAIL]'}")
    print("-" * 60)
    print(f"  OVERALL        : {'[PASS] Ready to proceed to nb06 / full_run' if overall else '[FAIL] Investigate before proceeding'}")
    print("=" * 60)
else:
    print("[WARN] No batch results to evaluate.")


ACCEPTANCE CRITERIA (BGL, 100-session smoke test)
  parse_success  : 100.0%  (threshold >= 96%)  [PASS]
  verify_passed  : 100.0%  (threshold >= 96%)  [PASS]
  LLM errors     : 0        (threshold = 0)      [PASS]
------------------------------------------------------------
  OVERALL        : [PASS] Ready to proceed to nb06 / full_run


## 11. Summary

### BGL Dataset Results

| Metric | Value |
|--------|-------|
| **Dataset** | BGL (Blue Gene/L Supercomputer) |
| **Total Log Lines** | 4,747,963 |
| **Train Sessions** | 284,884 |
| **Test Sessions** | 71,221 |
| **LLM** | Llama 3.1:8b (local via OpenAI) |
| **Evidence Store** | 332,359 documents (sessions + signatures) |
| **Retrieved Evidence** | Top-5 (4 anomaly + 1 normal) |

### Explanation Schema

Each explanation includes:
- **Signature**: Canonical error pattern name (e.g., `RAS_KERNEL_FATAL__DATA_STORAGE_INTERRUPT`)
- **Claims**: Typed assertions with evidence spans
  - `observation`: Direct facts from E0 (query session)
  - `pattern_match`: Similarity to historical anomalies (E1-E4)
  - `contrast`: Difference from normal sessions (E5)
- **Evidence Spans**: Line-level citations (e.g., `E0-L8`, `E1-L3`)

### Verification Checks

| Check | Description |
|-------|-------------|
| `structure` | Required fields present (prediction, summary, claims) |
| `evidence_ids` | All cited evidence IDs are valid |
| `evidence_coverage` | ≥80% of claims cite evidence |
| `keyword_match` | Claim keywords appear in cited evidence |
| `evidence_spans_validity` | Span references are valid (E0-L8 exists) |
| `signature` | Signature follows `COMPONENT__ERROR_TYPE` format |
| `span_keyword_match` | Keywords appear in specific cited lines |

### Key Design Decisions

1. **Mixed Retrieval**: 4 anomaly + 1 normal enables contrast claims
2. **E0 Convention**: Query session is always E0, retrieved evidence is E1-E5
3. **Line-Level Spans**: Claims cite specific lines, not just evidence IDs
4. **Signature Clustering**: Enables deduplication of similar anomalies